# 08 — Explainability with Grad-CAM
Modelin elin hangi bölgelerine baktığını görmek istiyoruz. Beklenen: büyüme plakaları (falanks ve radius/ulna uçları) ve bilek (karpal) kemikleri — radyologların da kemik yaşı belirlerken baktığı yerler.

In [ ]:
# On Colab: mount Drive and work from the project's notebooks folder
import os, sys
if "google.colab" in sys.modules and not os.getcwd().endswith("notebooks"):
    from google.colab import drive
    drive.mount("/content/drive")
    %cd /content/drive/MyDrive/bone-age/notebooks
%run 00_common.ipynb

In [ ]:
RUN = pd.read_json(OUTPUT_DIR / "experiments.json").query("runs.str.len() == 1").iloc[0]["runs"][0]
model, ckpt = load_model(RUN)
stats = ckpt["stats"]
df, _ = load_splits()
val = df[df["split"] == "val"].sort_values("boneage").reset_index(drop=True)
picks = val.iloc[np.linspace(0, len(val) - 1, 8).astype(int)].reset_index(drop=True)
ds = BoneAgeDataset(picks, get_transforms(False, ckpt["img_size"]), *stats)
print("Model:", RUN)

fig, axes = plt.subplots(2, len(picks), figsize=(2.6 * len(picks), 6))
for i in range(len(picks)):
    img, sex, _ = ds[i]
    cam, pred = grad_cam(model, img[None].to(DEVICE), sex[None].to(DEVICE))
    raw = img.permute(1, 2, 0).numpy()
    raw = (raw - raw.min()) / (raw.max() - raw.min())
    axes[0, i].imshow(raw)
    axes[0, i].set_title(f"true {picks.loc[i, 'boneage']:.0f} / pred {pred * stats[1] + stats[0]:.0f}")
    axes[1, i].imshow(raw)
    axes[1, i].imshow(cam, cmap="jet", alpha=0.45)
    for ax in axes[:, i]:
        ax.axis("off")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "gradcam.png", dpi=110, bbox_inches="tight")
plt.show()